# Predicting Moral Values in Text
### This Code offers predicting moral values from the MoralBERT weights deployad in Hugging Face.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    PretrainedConfig,
    PreTrainedModel
)
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    multilabel_confusion_matrix as mcm,
    precision_score,
    recall_score
)
from transformers.modeling_outputs import SequenceClassifierOutput

In [17]:
import os
import random
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from itertools import product
from scipy.special import softmax
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    multilabel_confusion_matrix as mcm,
    precision_score,
    recall_score
)
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from tabulate import tabulate
from torch.autograd import Function
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from tqdm import trange
from tqdm.auto import trange
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BertModel,
    PretrainedConfig,
    PreTrainedModel,
    get_linear_schedule_with_warmup
)
from transformers.modeling_outputs import SequenceClassifierOutput

In [3]:
base_model = "cahya/bert-base-indonesian-1.5G"
tokenizer = AutoTokenizer.from_pretrained(base_model)

HF_EXPORT_ROOT = Path("/content/drive/MyDrive/moralbert_cahya")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [4]:
from transformers import PretrainedConfig, PreTrainedModel, AutoModel, AutoConfig
from transformers.modeling_outputs import SequenceClassifierOutput
import torch
import torch.nn as nn

class PlainBERTConfig(PretrainedConfig):
    model_type = "plainbert"

    def __init__(
        self,
        base_model_name_or_path="cahya/bert-base-indonesian-1.5G",
        num_labels=2,
        class_weight=None,
        identity_weight=0.0,
        reconstruction_weight=0.0,
        moral_weight=1.0,
        freeze_bert=False,
        id2label=None,
        label2id=None,
        **kwargs,
    ):
        super().__init__(num_labels=num_labels, id2label=id2label, label2id=label2id, **kwargs)
        self.base_model_name_or_path = base_model_name_or_path
        self.num_labels = num_labels
        self.class_weight = class_weight if class_weight is not None else [1.0] * num_labels
        self.identity_weight = float(identity_weight)
        self.reconstruction_weight = float(reconstruction_weight)
        self.moral_weight = float(moral_weight)
        self.freeze_bert = bool(freeze_bert)
        self.problem_type = "single_label_classification"


class PlainBERTForSequenceClassification(PreTrainedModel):
    config_class = PlainBERTConfig
    base_model_prefix = "bert"
    supports_gradient_checkpointing = False

    def __init__(self, config):
        super().__init__(config)

        self.num_labels = config.num_labels
        self.freeze = config.freeze_bert

        base_encoder_config = AutoConfig.from_pretrained(config.base_model_name_or_path)
        self.bert = AutoModel.from_config(base_encoder_config)
        bert_dim = self.bert.config.hidden_size

        self.invariant_trans = nn.Linear(bert_dim, bert_dim)

        if config.identity_weight + config.reconstruction_weight == 0:
            self.moral_classification = nn.Linear(bert_dim, config.num_labels)
        else:
            self.moral_classification = nn.Sequential(
                nn.Linear(bert_dim, bert_dim),
                nn.ReLU(),
                nn.Linear(bert_dim, config.num_labels),
            )

        class_weight = config.class_weight if config.class_weight is not None else [1.0] * config.num_labels
        if class_weight and class_weight[0] > 0:
            weights = torch.tensor(class_weight).float()
        else:
            weights = torch.ones(config.num_labels).float()
        self.register_buffer("class_weights", weights)

        self.reconstruction_feed = nn.Linear(bert_dim, bert_dim)
        self.loss_reconstruction = nn.MSELoss()
        self.register_buffer("identity", torch.eye(bert_dim))

        try:
            self.post_init()
        except Exception as e:
            print("Warning during post_init:", e)

    def forward(
        self,
        input_ids=None,
        token_type_ids=None,
        attention_mask=None,
        labels=None,
        original_bert_embeddings=None,
        **kwargs,
    ):
        if self.freeze:
            with torch.no_grad():
                cls = self.bert(
                    input_ids=input_ids,
                    token_type_ids=token_type_ids,
                    attention_mask=attention_mask
                ).last_hidden_state[:, 0, :]
        else:
            cls = self.bert(
                input_ids=input_ids,
                token_type_ids=token_type_ids,
                attention_mask=attention_mask
            ).last_hidden_state[:, 0, :]

        z = self.invariant_trans(cls)
        logits = self.moral_classification(z)

        total_loss = None
        if labels is not None:
            loss_fn_moral = nn.CrossEntropyLoss(weight=self.class_weights)
            loss_moral = loss_fn_moral(logits, labels)

            if original_bert_embeddings is not None and self.config.reconstruction_weight > 0:
                loss_recon = self.loss_reconstruction(
                    self.reconstruction_feed(z),
                    original_bert_embeddings
                ) * self.config.reconstruction_weight
            else:
                loss_recon = 0.0

            if self.config.identity_weight > 0:
                loss_identity = torch.norm(
                    self.invariant_trans.weight - self.identity
                ) * self.config.identity_weight
            else:
                loss_identity = 0.0

            total_loss = (loss_moral * self.config.moral_weight) + loss_recon + loss_identity

        return SequenceClassifierOutput(loss=total_loss, logits=logits)

In [5]:
def preprocessing(input_text, tokenizer):
    return tokenizer(
        input_text,
        add_special_tokens=True,
        max_length=150,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="pt"
    )

In [6]:
possible_labels = [
    "kepedulian", "menyakiti",
    "keadilan", "kecurangan",
    "loyalitas", "pengkhianatan",
    "otoritas", "pembangkangan",
    "kesucian", "kemerosotan",
    "kebebasan", "penindasan"
]

In [7]:
for lab in possible_labels:
    checkpoint_folder = HF_EXPORT_ROOT / lab.replace(" ", "_")
    print(lab, "->", checkpoint_folder)
    print("exists:", checkpoint_folder.exists(), "| is_dir:", checkpoint_folder.is_dir())

kepedulian -> /content/drive/MyDrive/moralbert_cahya/kepedulian
exists: True | is_dir: True
menyakiti -> /content/drive/MyDrive/moralbert_cahya/menyakiti
exists: True | is_dir: True
keadilan -> /content/drive/MyDrive/moralbert_cahya/keadilan
exists: True | is_dir: True
kecurangan -> /content/drive/MyDrive/moralbert_cahya/kecurangan
exists: True | is_dir: True
loyalitas -> /content/drive/MyDrive/moralbert_cahya/loyalitas
exists: True | is_dir: True
pengkhianatan -> /content/drive/MyDrive/moralbert_cahya/pengkhianatan
exists: True | is_dir: True
otoritas -> /content/drive/MyDrive/moralbert_cahya/otoritas
exists: True | is_dir: True
pembangkangan -> /content/drive/MyDrive/moralbert_cahya/pembangkangan
exists: True | is_dir: True
kesucian -> /content/drive/MyDrive/moralbert_cahya/kesucian
exists: True | is_dir: True
kemerosotan -> /content/drive/MyDrive/moralbert_cahya/kemerosotan
exists: True | is_dir: True
kebebasan -> /content/drive/MyDrive/moralbert_cahya/kebebasan
exists: True | is_di

In [8]:
import os
import __main__

dummy_py_path = os.path.join(os.getcwd(), "notebook_session.py")
if not os.path.exists(dummy_py_path):
    with open(dummy_py_path, "w", encoding="utf-8") as f:
        f.write("# dummy file for transformers custom model in notebook\n")

__main__.__file__ = dummy_py_path

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loaded_models = {}

for lab in possible_labels:
    checkpoint_folder = HF_EXPORT_ROOT / lab.replace(" ", "_")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = PlainBERTForSequenceClassification.from_pretrained(
        str(checkpoint_folder),
        local_files_only=True
    ).to(device)

    model.eval()
    loaded_models[lab] = model

print("All models loaded successfully.")
print(checkpoint_folder)
print(os.listdir(checkpoint_folder))

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

All models loaded successfully.
/content/drive/MyDrive/moralbert_cahya/penindasan
['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'configuration_plainbert.py', 'modeling_plainbert.py', 'README.md']


In [10]:
def get_model_score(sentence, label_name):
    model = loaded_models[label_name]

    encodeds = preprocessing(sentence, tokenizer)
    encodeds = {k: v.to(device) for k, v in encodeds.items()}

    with torch.no_grad():
        outputs = model(**encodeds)
        probs = F.softmax(outputs.logits, dim=1)

    score_positive = probs[0, 1].item()
    score_negative = probs[0, 0].item()

    return {
        "score_positive": score_positive,
        "score_negative": score_negative,
        "pred_label": int(score_positive >= 0.5)
    }

In [11]:
test_df = pd.read_csv("test_indonesia.csv")
sentences = test_df["sentence"].tolist()

In [12]:
results = []

for sentence in sentences:
    row = {"sentence": sentence}

    for lab in possible_labels:
        output = get_model_score(sentence, lab)
        row[f"{lab}_score"] = output["score_positive"]
        #row[f"{lab}_pred"] = output["pred_label"]

    results.append(row)

results_df = pd.DataFrame(results)
results_df

,sentence,kepedulian_score,menyakiti_score,keadilan_score,kecurangan_score,loyalitas_score,pengkhianatan_score,otoritas_score,pembangkangan_score,kesucian_score,kemerosotan_score,kebebasan_score,penindasan_score
0,Ia seorang petani yang rajin bekerja walaupun ...,0.000958,0.001727,0.000461,0.000224,0.000278,0.003677,0.000864,0.001957,0.000400,0.002963,0.000170,0.006427
1,Ia bisa mencukupi kebutuhannya dari hasil kerj...,0.001028,0.001670,0.000409,0.000203,0.000281,0.003917,0.000895,0.001972,0.000390,0.002402,0.000162,0.006186
2,Aku akan bersediamenemanimu jika kau tidak jad...,0.000996,0.001732,0.000400,0.000192,0.000257,0.004479,0.002371,0.001970,0.000794,0.002462,0.000187,0.006096
3,"“Dia mungkin bidadari yang turun dari langit,”...",0.000892,0.001718,0.000580,0.000190,0.000251,0.003614,0.000894,0.001981,0.000494,0.002476,0.000147,0.005258
4,"Karena ketekunan dan keuletannya, petani itu h...",0.000968,0.001740,0.000461,0.000206,0.000352,0.003974,0.000941,0.002015,0.000439,0.002417,0.000228,0.006328
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4645,"Terima kasih kunang-kunang,” ujar Sidang Belawan.",0.001052,0.002094,0.000444,0.000191,0.000257,0.003513,0.001374,0.002002,0.001098,0.002232,0.000143,0.005134
4646,Sepertinya kau memang berjodoh dengan adik kami.,0.001034,0.001757,0.000399,0.000189,0.000244,0.003797,0.000921,0.001975,0.000382,0.002184,0.000146,0.005269
4647,"Jaga dan rawatlah dia dengan baik,” pesan sala...",0.000858,0.001637,0.000432,0.000189,0.000293,0.003757,0.001066,0.001974,0.000758,0.002543,0.000152,0.005777
4648,"Semenjak kejadian itu, tidak ada lagi kesalahp...",0.000943,0.004687,0.000496,0.000207,0.001262,0.003904,0.003126,0.002038,0.000729,0.002716,0.000141,0.005589


In [13]:
input_files = test_df["sentence"].values

tokenizer = AutoTokenizer.from_pretrained(base_model)

original_input_id = []
original_attention_masks = []
original_token_type_id = []

def preprocessing(input_text, tokenizer):
    return tokenizer(
        input_text,
        add_special_tokens=True,
        max_length=150,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

for sample in input_files:
    original_encoding_dict = preprocessing(sample, tokenizer)

    original_input_id.append(original_encoding_dict["input_ids"])
    original_attention_masks.append(original_encoding_dict["attention_mask"])

    if "token_type_ids" in original_encoding_dict:
        original_token_type_id.append(original_encoding_dict["token_type_ids"])
    else:
        original_token_type_id.append(torch.zeros_like(original_encoding_dict["input_ids"]))

original_input_id = torch.cat(original_input_id, dim=0)
original_token_type_id = torch.cat(original_token_type_id, dim=0)
original_attention_masks = torch.cat(original_attention_masks, dim=0)

print("recreated")
print("original_input_id.is_meta =", original_input_id.is_meta)
print("original_token_type_id.is_meta =", original_token_type_id.is_meta)
print("original_attention_masks.is_meta =", original_attention_masks.is_meta)

recreated
original_input_id.is_meta = False
original_token_type_id.is_meta = False
original_attention_masks.is_meta = False


In [18]:
possible_labels = [
    "kepedulian", "menyakiti",
    "keadilan", "kecurangan",
    "loyalitas", "pengkhianatan",
    "otoritas", "pembangkangan",
    "kesucian", "kemerosotan",
    "kebebasan", "penindasan"
]

batch_size = 16

torch.set_default_device("cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

predictions = []
threshold_summary = []

for lab_idx, lab in enumerate(possible_labels):
    print(f"\nProcessing label: {lab}")

    best_f1 = 0
    best_th = 0.5
    best_y = None

    true_labels_for_lab = test_df[lab].astype(int).tolist()

    val_set = TensorDataset(
        original_input_id,
        original_token_type_id,
        original_attention_masks,
        torch.tensor(true_labels_for_lab, dtype=torch.long)
    )

    validation_dataloader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

    checkpoint_folder = HF_EXPORT_ROOT / lab.replace(" ", "_")

    model = PlainBERTForSequenceClassification.from_pretrained(
        str(checkpoint_folder),
        local_files_only=True
    )
    model = model.to(device)
    model.eval()

    all_probs = []
    y_true = []

    for batch in validation_dataloader:
        b_input_ids, b_token_type_ids, b_input_mask, b_labels = [
            t.to(device) for t in batch
        ]

        model_inputs = {
            "input_ids": b_input_ids,
            "attention_mask": b_input_mask
        }

        if b_token_type_ids is not None:
            model_inputs["token_type_ids"] = b_token_type_ids

        with torch.no_grad():
            outputs = model(**model_inputs)

            logits = outputs.logits.detach().cpu()
            probs = torch.softmax(logits, dim=1).numpy()[:, 1]

            all_probs.extend(probs.tolist())
            y_true.extend(b_labels.detach().cpu().numpy().tolist())

    all_probs = np.array(all_probs)
    y_true = np.array(y_true)

    for th in np.arange(0.05, 1.00, 0.05):
        y_pred = (all_probs >= th).astype(int)
        f1 = f1_score(y_true, y_pred, average="binary", zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_th = th
            best_y = y_pred.copy()

    threshold_summary.append({
        "label": lab,
        "best_threshold": float(best_th),
        "best_f1": float(best_f1)
    })

    if lab_idx == 0:
        for ex_id, (pred_label, true_label) in enumerate(zip(best_y, y_true)):
            predictions.append({
                "id": ex_id,
                f"pred_{lab}": int(pred_label),
                f"true_{lab}": int(true_label)
            })
    else:
        for ex_id, (pred_label, true_label) in enumerate(zip(best_y, y_true)):
            predictions[ex_id][f"pred_{lab}"] = int(pred_label)
            predictions[ex_id][f"true_{lab}"] = int(true_label)

    print("Evaluation")
    print(f"{lab} | best threshold: {best_th:.2f} | best F1: {best_f1:.4f}")

    target_names = [f"Non-{lab}", lab]
    report = classification_report(
        y_true,
        best_y,
        target_names=target_names,
        zero_division=0
    )

    print("\nClassification Report:")
    print(report)

pred_df = pd.DataFrame(predictions)
threshold_df = pd.DataFrame(threshold_summary)


Processing label: kepedulian


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Evaluation
kepedulian | best threshold: 0.95 | best F1: 0.9891

Classification Report:
                precision    recall  f1-score   support

Non-kepedulian       1.00      1.00      1.00      4558
    kepedulian       0.99      0.99      0.99        92

      accuracy                           1.00      4650
     macro avg       0.99      0.99      0.99      4650
  weighted avg       1.00      1.00      1.00      4650


Processing label: menyakiti


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Evaluation
menyakiti | best threshold: 0.90 | best F1: 0.9326

Classification Report:
               precision    recall  f1-score   support

Non-menyakiti       1.00      1.00      1.00      4563
    menyakiti       0.91      0.95      0.93        87

     accuracy                           1.00      4650
    macro avg       0.96      0.98      0.97      4650
 weighted avg       1.00      1.00      1.00      4650


Processing label: keadilan


...

Evaluation
kemerosotan | best threshold: 0.50 | best F1: 0.9268

Classification Report:
                 precision    recall  f1-score   support

Non-kemerosotan       1.00      1.00      1.00      4629
    kemerosotan       0.95      0.90      0.93        21

       accuracy                           1.00      4650
      macro avg       0.97      0.95      0.96      4650
   weighted avg       1.00      1.00      1.00      4650


Processing label: kebebasan


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Evaluation
kebebasan | best threshold: 0.40 | best F1: 0.9500

Classification Report:
               precision    recall  f1-score   support

Non-kebebasan       1.00      1.00      1.00      4630
    kebebasan       0.95      0.95      0.95        20

     accuracy                           1.00      4650
    macro avg       0.97      0.97      0.97      4650
 weighted avg       1.00      1.00      1.00      4650


Processing label: penindasan


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

Evaluation
penindasan | best threshold: 0.45 | best F1: 0.8966

Classification Report:
                precision    recall  f1-score   support

Non-penindasan       1.00      1.00      1.00      4636
    penindasan       0.87      0.93      0.90        14

      accuracy                           1.00      4650
     macro avg       0.93      0.96      0.95      4650
  weighted avg       1.00      1.00      1.00      4650



In [19]:
pd.set_option('display.max_columns', None)

In [20]:
results = []

for idx_lab, lab in enumerate(possible_labels):
    result = {"Moral Value": lab}
    true = test_df[lab].values
    candidate = pred_df["pred_" + lab].values

    result["F1 Score (Binary)"] = f1_score(true, candidate, average="binary", zero_division=0)
    result["F1 Score (Weighted)"] = f1_score(true, candidate, average="weighted", zero_division=0)

    result["Precision Score (Binary)"] = precision_score(true, candidate, average="binary", zero_division=0)
    result["Precision Score (Weighted)"] = precision_score(true, candidate, average="weighted", zero_division=0)

    result["Recall Score (Binary)"] = recall_score(true, candidate, average="binary", zero_division=0)
    result["Recall Score (Weighted)"] = recall_score(true, candidate, average="weighted", zero_division=0)

    result["Accuracy"] = accuracy_score(true, candidate)

    results.append(result)

results = pd.DataFrame(results)

In [21]:
possible_labels = [
    "kepedulian", "menyakiti",
    "keadilan", "kecurangan",
    "loyalitas", "pengkhianatan",
    "otoritas", "pembangkangan",
    "kesucian", "kemerosotan",
    "kebebasan", "penindasan"
]

test_df.reset_index(drop=True, inplace=True)
n_bootstrap_iters = 1000

bootstrap_results = {
    label: {
        metric: [] for metric in [
            "F1 (Binary)", "F1 (Macro)", "F1 (Weighted)",
            "Precision (Binary)", "Precision (Macro)", "Precision (Weighted)",
            "Recall (Binary)", "Recall (Macro)", "Recall (Weighted)",
            "Accuracy"
        ]
    } for label in possible_labels
}

for _ in range(n_bootstrap_iters):
    for lab in possible_labels:
        sample_indices = resample(np.arange(len(test_df)), replace=True)
        true = test_df.loc[sample_indices, lab].values
        candidate = pred_df.loc[sample_indices, f"pred_{lab}"].values

        bootstrap_results[lab]["F1 (Binary)"].append(f1_score(true, candidate, average="binary", zero_division=0))
        bootstrap_results[lab]["F1 (Macro)"].append(f1_score(true, candidate, average="macro", zero_division=0))
        bootstrap_results[lab]["F1 (Weighted)"].append(f1_score(true, candidate, average="weighted", zero_division=0))
        bootstrap_results[lab]["Precision (Binary)"].append(precision_score(true, candidate, average="binary", zero_division=0))
        bootstrap_results[lab]["Precision (Macro)"].append(precision_score(true, candidate, average="macro", zero_division=0))
        bootstrap_results[lab]["Precision (Weighted)"].append(precision_score(true, candidate, average="weighted", zero_division=0))
        bootstrap_results[lab]["Recall (Binary)"].append(recall_score(true, candidate, average="binary", zero_division=0))
        bootstrap_results[lab]["Recall (Macro)"].append(recall_score(true, candidate, average="macro", zero_division=0))
        bootstrap_results[lab]["Recall (Weighted)"].append(recall_score(true, candidate, average="weighted", zero_division=0))
        bootstrap_results[lab]["Accuracy"].append(accuracy_score(true, candidate))

std_devs = {
    label: {metric: np.std(values) for metric, values in metrics.items()}
    for label, metrics in bootstrap_results.items()
}

final_results = []
for lab in possible_labels:
    result = {"Moral Value": lab}
    true = test_df[lab].values
    candidate = pred_df[f"pred_{lab}"].values

    result["F1 Score (Binary)"] = f"{f1_score(true, candidate, average='binary', zero_division=0):.2f} ± {std_devs[lab]['F1 (Binary)']:.2f}"
    result["F1 Score (Macro)"] = f"{f1_score(true, candidate, average='macro', zero_division=0):.2f} ± {std_devs[lab]['F1 (Macro)']:.2f}"
    result["F1 Score (Weighted)"] = f"{f1_score(true, candidate, average='weighted', zero_division=0):.2f} ± {std_devs[lab]['F1 (Weighted)']:.2f}"

    result["Precision Score (Binary)"] = f"{precision_score(true, candidate, average='binary', zero_division=0):.2f} ± {std_devs[lab]['Precision (Binary)']:.2f}"
    result["Precision Score (Macro)"] = f"{precision_score(true, candidate, average='macro', zero_division=0):.2f} ± {std_devs[lab]['Precision (Macro)']:.2f}"
    result["Precision Score (Weighted)"] = f"{precision_score(true, candidate, average='weighted', zero_division=0):.2f} ± {std_devs[lab]['Precision (Weighted)']:.2f}"

    result["Recall Score (Binary)"] = f"{recall_score(true, candidate, average='binary', zero_division=0):.2f} ± {std_devs[lab]['Recall (Binary)']:.2f}"
    result["Recall Score (Macro)"] = f"{recall_score(true, candidate, average='macro', zero_division=0):.2f} ± {std_devs[lab]['Recall (Macro)']:.2f}"
    result["Recall Score (Weighted)"] = f"{recall_score(true, candidate, average='weighted', zero_division=0):.2f} ± {std_devs[lab]['Recall (Weighted)']:.2f}"

    result["Accuracy"] = f"{accuracy_score(true, candidate):.2f} ± {std_devs[lab]['Accuracy']:.2f}"

    final_results.append(result)

results_df = pd.DataFrame(final_results)
results_df

,Moral Value,F1 Score (Binary),F1 Score (Macro),F1 Score (Weighted),Precision Score (Binary),Precision Score (Macro),Precision Score (Weighted),Recall Score (Binary),Recall Score (Macro),Recall Score (Weighted),Accuracy
0,kepedulian,0.99 ± 0.01,0.99 ± 0.00,1.00 ± 0.00,0.99 ± 0.01,0.99 ± 0.01,1.00 ± 0.00,0.99 ± 0.01,0.99 ± 0.01,1.00 ± 0.00,1.00 ± 0.00
1,menyakiti,0.93 ± 0.02,0.97 ± 0.01,1.00 ± 0.00,0.91 ± 0.03,0.96 ± 0.01,1.00 ± 0.00,0.95 ± 0.02,0.98 ± 0.01,1.00 ± 0.00,1.00 ± 0.00
2,keadilan,0.81 ± 0.04,0.90 ± 0.02,0.99 ± 0.00,0.84 ± 0.04,0.92 ± 0.02,0.99 ± 0.00,0.78 ± 0.05,0.89 ± 0.02,0.99 ± 0.00,0.99 ± 0.00
3,kecurangan,0.97 ± 0.03,0.99 ± 0.01,1.00 ± 0.00,0.94 ± 0.05,0.97 ± 0.03,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00
4,loyalitas,0.81 ± 0.06,0.90 ± 0.03,1.00 ± 0.00,0.84 ± 0.08,0.92 ± 0.04,1.00 ± 0.00,0.78 ± 0.08,0.89 ± 0.04,1.00 ± 0.00,1.00 ± 0.00
5,pengkhianatan,0.99 ± 0.01,0.99 ± 0.01,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00,0.97 ± 0.03,0.99 ± 0.01,1.00 ± 0.00,1.00 ± 0.00
6,otoritas,0.95 ± 0.01,0.97 ± 0.01,1.00 ± 0.00,0.92 ± 0.02,0.96 ± 0.01,1.00 ± 0.00,0.99 ± 0.01,0.99 ± 0.00,1.00 ± 0.00,1.00 ± 0.00
7,pembangkangan,0.92 ± 0.05,0.96 ± 0.02,1.00 ± 0.00,0.95 ± 0.05,0.97 ± 0.03,1.00 ± 0.00,0.90 ± 0.07,0.95 ± 0.03,1.00 ± 0.00,1.00 ± 0.00
8,kesucian,0.73 ± 0.04,0.86 ± 0.02,0.99 ± 0.00,0.75 ± 0.05,0.87 ± 0.02,0.99 ± 0.00,0.70 ± 0.05,0.85 ± 0.02,0.99 ± 0.00,0.99 ± 0.00
9,kemerosotan,0.93 ± 0.04,0.96 ± 0.02,1.00 ± 0.00,0.95 ± 0.05,0.97 ± 0.03,1.00 ± 0.00,0.90 ± 0.07,0.95 ± 0.03,1.00 ± 0.00,1.00 ± 0.00
